In [1]:
import pandas as pd
import re

df = pd.read_csv("data/final_labels_SG2.csv", sep=";")
print(df.shape) # rows and coulmns


(3674, 8)


In [6]:

print(df.isna().sum())

print(df["label_bias"].value_counts(dropna=False))


text                0
news_link          32
outlet              0
topic               0
type             1000
label_bias          0
label_opinion       0
biased_words        0
dtype: int64
label_bias
Non-biased      1863
Biased          1810
No agreement       1
Name: count, dtype: int64


In [5]:
print("Duplicate sentences:", df.duplicated(subset=["text"]).sum())

print("\nTop 10 most frequent article links:")
print(df["news_link"].value_counts().head(10))

Duplicate sentences: 0

Top 10 most frequent article links:
news_link
http://feedproxy.google.com/~r/breitbart/~3/2RH0NVHEl6Q/                                                                                                       15
http://feedproxy.google.com/~r/breitbart/~3/MBTkPPu9uSo/                                                                                                       15
https://www.alternet.org/2020/06/law-and-order-is-a-debased-concept-used-to-cover-up-right-wing-crime-and-depravity-heres-why/                                 12
http://feedproxy.google.com/~r/breitbart/~3/ei0CD392IU0/                                                                                                       12
http://feedproxy.google.com/~r/breitbart/~3/_MdFljKx8cY/                                                                                                       11
https://www.alternet.org/2020/06/bunker-boy-trump-thinks-he-can-edit-reality-like-tv-show/                              

In [10]:

clean_df = df.dropna(subset=["text"]).copy()
clean_df = clean_df[clean_df["label_bias"].isin(["Biased", "Non-biased"])].reset_index(drop=True)

clean_df["type"] = clean_df["type"].fillna("Unknown")

In [13]:
missing_links = clean_df["news_link"].isna()
clean_df.loc[missing_links, "news_link"] = [
    f"unknown_link_{i}" for i in range(missing_links.sum())
]

In [15]:
clean_df["label"] = clean_df["label_bias"].map({"Biased": 1, "Non-biased": 0})
clean_df["original_text"] = clean_df["text"]

In [18]:
def preprocess_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)      # remove any stray HTML tags
    text = re.sub(r"\s+", " ", text).strip()  # collapse multiple spaces/newlines
    return text

clean_df["model_text"] = clean_df["original_text"].apply(preprocess_text)

In [20]:
#Quick verification
print("Remaining rows:", len(clean_df))
print("Remaining nulls:\n", clean_df.isna().sum())
clean_df[["original_text", "model_text", "label"]].head(3)

Remaining rows: 3673
Remaining nulls:
 text             0
news_link        0
outlet           0
topic            0
type             0
label_bias       0
label_opinion    0
biased_words     0
label            0
original_text    0
model_text       0
dtype: int64


,original_text,model_text,label
0,"""Orange Is the New Black"" star Yael Stone is r...","""orange is the new black"" star yael stone is r...",0
1,"""We have one beautiful law,"" Trump recently sa...","""we have one beautiful law,"" trump recently sa...",1
2,"...immigrants as criminals and eugenics, all o...","...immigrants as criminals and eugenics, all o...",1


In [21]:
clean_df["label"].value_counts(normalize = True)

label
0    0.507215
1    0.492785
Name: proportion, dtype: float64

In [26]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# Grab the first fold: 80% train indices, 20% test indices
train_idx, test_idx = next(
    sgkf.split(X=clean_df, y=clean_df["label"], groups=clean_df["news_link"])
)

train_df = clean_df.iloc[train_idx].reset_index(drop=True)
test_df = clean_df.iloc[test_idx].reset_index(drop=True)

# 1. Verify zero article leakage between train and test
shared_links = set(train_df["news_link"]).intersection(set(test_df["news_link"]))
print("Shared article links between train & test:", len(shared_links))

# 2. Verify sizes and class balance in both splits
print(f"Train rows: {len(train_df)} ({len(train_df)/len(clean_df):.1%})")
print(f"Test rows:  {len(test_df)} ({len(test_df)/len(clean_df):.1%})")
print("\nTrain label proportions:\n", train_df["label"].value_counts(normalize=True).round(4))
print("\nTest label proportions:\n", test_df["label"].value_counts(normalize=True).round(4))

# 3. Export locked datasets for Member 2 & Member 3
clean_df.to_csv("data/babe_clean.csv", index=False)
train_df.to_csv("data/train.csv", index=False)
test_df.to_csv("data/test.csv", index=False)

Shared article links between train & test: 0
Train rows: 2938 (80.0%)
Test rows:  735 (20.0%)

Train label proportions:
 label
0    0.5071
1    0.4929
Name: proportion, dtype: float64

Test label proportions:
 label
0    0.5075
1    0.4925
Name: proportion, dtype: float64
